In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ================================
# Falcon-7B-Instruct (native) + LoRA — Inference-Only
# No HF login/token required (public models/adapters only)
# Colab/T4-friendly (defaults to 4-bit)
# ================================
# Installs
!pip -q uninstall -y transformers accelerate || true
!pip -q install "transformers>=4.45.0" "accelerate>=0.34.2" "bitsandbytes>=0.43.1" peft "huggingface_hub>=0.23" sentencepiece

# Purge legacy remote Falcon code so Transformers uses native implementation
import shutil, os
from pathlib import Path
legacy_mod_dir = Path.home() / ".cache/huggingface/modules/transformers_modules/tiiuae/falcon-7b-instruct"
shutil.rmtree(legacy_mod_dir, ignore_errors=True)

# ---------------- Config (EDIT THESE) ----------------
FALCON_ID        = "tiiuae/falcon-7b-instruct"   # native Transformers path, public
LORA_ADAPTER_ID  = ""  # e.g. "your-user/falcon7b-instruct-lora" OR a local folder path like "/content/my_lora"
MERGE_LORA       = False  # True=fuse adapter into base (slightly faster, more VRAM)

# Quantization (default = 4-bit for stability on T4)
USE_4BIT = True
USE_8BIT = False  # If you set this True, see note below re: CPU offload error

MAX_NEW_TOKENS = 384

# ---------------- Load Model + LoRA ----------------
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline
from peft import PeftModel

torch.backends.cuda.matmul.allow_tf32 = True  # small perf tweak

def _bnb_4bit():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )

def _bnb_8bit_offload():  # only if you insist on 8-bit and allow CPU offload
    return BitsAndBytesConfig(
        load_in_8bit=True,
        llm_int8_enable_fp32_cpu_offload=True
    )

_GEN = None

def load_falcon_lora_pipeline(
    model_id=FALCON_ID,
    lora_id=LORA_ADAPTER_ID,
    merge_lora=MERGE_LORA,
    max_new_tokens=MAX_NEW_TOKENS
):
    global _GEN
    if _GEN is not None:
        return _GEN

    if USE_4BIT:
        quant_cfg = _bnb_4bit()
        max_memory = None
    elif USE_8BIT:
        quant_cfg = _bnb_8bit_offload()
        # Be explicit with offload limits if needed (optional):
        max_memory = {0: "13GiB", "cpu": "48GiB"}
    else:
        quant_cfg = None
        max_memory = None

    torch.cuda.empty_cache()

    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=quant_cfg,
        device_map="auto",
        max_memory=max_memory,
        low_cpu_mem_usage=True,
        torch_dtype=torch.float16,
        attn_implementation="eager",   # consistent KV cache on consumer GPUs
        trust_remote_code=False        # native Falcon implementation
    )

    tok = AutoTokenizer.from_pretrained(model_id, use_fast=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token

    # Attach LoRA if provided (public HF repo or local folder path)
    if lora_id and lora_id.strip():
        print(f"[LoRA] Loading adapter from: {lora_id}")
        model = PeftModel.from_pretrained(model, lora_id)
        if merge_lora:
            print("[LoRA] merge_and_unload=True (fusing into base weights)")
            model = model.merge_and_unload()

    # Generation hygiene
    model.config.use_cache = True
    try:
        model.generation_config.pad_token_id = tok.pad_token_id
        model.generation_config.eos_token_id = tok.eos_token_id
    except Exception:
        pass

    _GEN = pipeline(
        "text-generation",
        model=model,
        tokenizer=tok,
        max_new_tokens=max_new_tokens,
        do_sample=False,           # deterministic; add sampling later if desired
        repetition_penalty=1.1
    )
    return _GEN

# ---------------- Simple Inference Helpers ----------------
def generate(prompt: str, max_new_tokens: int = MAX_NEW_TOKENS, do_sample: bool = False, temperature: float = 0.7, top_p: float = 0.9):
    """
    If do_sample=False -> temperature/top_p are ignored (deterministic).
    Flip do_sample=True to enable sampling.
    """
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)
    if do_sample:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature, top_p=top_p, repetition_penalty=1.1)
    else:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    # pipeline returns full prompt+completion for text-generation
    return out[0]["generated_text"][len(prompt):].strip()

# ---------------- Usage Examples ----------------
# 1) Base Falcon (no LoRA): leave LORA_ADAPTER_ID = ""
# txt = generate("Write a 2-sentence summary of the importance of unit testing in software engineering:")

# 2) Falcon + LoRA:
# - Public HF LoRA: set LORA_ADAPTER_ID = "your-user/falcon7b-instruct-lora"
# - OR Local LoRA:  set LORA_ADAPTER_ID = "/content/my_lora_dir" (must contain adapter_config.json, adapter_model.bin/safetensors)
# Then:
# txt = generate("Summarize the benefits of LoRA fine-tuning for instruction-following models:")
# print(txt)

print("Ready. Set LORA_ADAPTER_ID if you want LoRA; call generate(prompt).")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 92.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 MB 11.8 MB/s eta 0:00:00
Ready. Set LORA_ADAPTER_ID if you want LoRA; call generate(prompt).


In [ ]:
# ---------------- Evaluation: 6 Metrics ----------------
from statistics import mean
import numpy as np

# Example eval queries and gold answers (edit for your own domain)
EVAL_QUERIES = [
    "What is the importance of unit testing in software engineering?",
    "List two best practices for clean code.",
]

GOLD = {
    "What is the importance of unit testing in software engineering?":
        "Unit testing helps detect bugs early, improves code reliability, and supports safe refactoring."
}

def evaluate_falcon_lora(k: int = 1, max_new_tokens: int = MAX_NEW_TOKENS):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)

    latencies = []
    grounded, relevant, coverage, lengths, rouge_scores = [], [], [], [], []

    for q in EVAL_QUERIES:
        t0 = time.time()
        out = gen(q, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
        t1 = time.time()
        ans = out[0]["generated_text"][len(q):].strip()

        # metrics
        g = groundedness(ans, [ ])     # here [] since no retriever context — can adapt if you add RAG
        r = relevance(q, ans)
        c = context_coverage(ans, [ ])
        L = len(ans.split())
        rl = rougeL_f1(ans, GOLD[q]) if q in GOLD else None

        latencies.append(t1 - t0)
        grounded.append(g); relevant.append(r); coverage.append(c); lengths.append(L)
        if rl is not None: rouge_scores.append(rl)

        print(f"\nQ: {q}\nA: {ans}\n")

    def fmt(x, s):
        try: return format(x, s)
        except: return "None"

    print("\n=== Evaluation Report ===")
    print("groundedness_avg:     ", fmt(mean(grounded), ".3f"))
    print("relevance_avg:        ", fmt(mean(relevant), ".3f"))
    print("context_coverage_avg: ", fmt(mean(coverage), ".3f"))
    print("rougeL_f1_avg_on_gold:", (fmt(mean(rouge_scores), ".3f") if rouge_scores else "-"))
    print("answer_len_avg:       ", fmt(mean(lengths), ".1f"))
    print("latency p50 / p95:    ",
          fmt(np.percentile(latencies, 50), ".2f")+"s", "/",
          fmt(np.percentile(latencies, 95), ".2f")+"s")


In [ ]:
import time
# ---------------- Evaluation: 6 Metrics ----------------
import time
import numpy as np
from statistics import mean

# Example eval queries and gold answers (edit these for your domain)
EVAL_QUERIES = [
    "What is the importance of unit testing in software engineering?",
    "List two best practices for clean code."
]

GOLD = {
    "What is the importance of unit testing in software engineering?":
        "Unit testing helps detect bugs early, improves code reliability, and supports safe refactoring."
}

def evaluate_falcon_lora(k: int = 1, max_new_tokens: int = MAX_NEW_TOKENS):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)

    latencies = []
    grounded, relevant, coverage, lengths, rouge_scores = [], [], [], [], []

    for q in EVAL_QUERIES:
        t0 = time.time()
        out = gen(q, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
        t1 = time.time()
        ans = out[0]["generated_text"][len(q):].strip()

        # metrics (groundedness/context_coverage dummy=0.0 since no RAG context)
        g = 0.0
        r = relevance(q, ans)
        c = 0.0
        L = len(ans.split())
        rl = rougeL_f1(ans, GOLD[q]) if q in GOLD else None

        latencies.append(t1 - t0)
        grounded.append(g); relevant.append(r); coverage.append(c); lengths.append(L)
        if rl is not None: rouge_scores.append(rl)

        print(f"\nQ: {q}\nA: {ans}\n")

    def fmt(x, s):
        try: return format(x, s)
        except: return "None"

    print("\n=== Evaluation Report ===")
    print("groundedness_avg:     ", fmt(mean(grounded), ".3f"))
    print("relevance_avg:        ", fmt(mean(relevant), ".3f"))
    print("context_coverage_avg: ", fmt(mean(coverage), ".3f"))
    print("rougeL_f1_avg_on_gold:", (fmt(mean(rouge_scores), ".3f") if rouge_scores else "-"))
    print("answer_len_avg:       ", fmt(mean(lengths), ".1f"))
    p50s = (fmt(np.percentile(latencies, 50), ".2f")+"s") if latencies else "None"
    p95s = (fmt(np.percentile(latencies, 95), ".2f")+"s") if latencies else "None"
    print("latency p50 / p95:    ", p50s, "/", p95s)



In [ ]:
!pip install rouge-score

from sentence_transformers import SentenceTransformer, util as st_util
from rouge_score import rouge_scorer
import re

# Init emb + rouge scorer
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _normalize(s: str):
    s = s.lower()
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def relevance(query: str, answer: str) -> float:
    embs = embedder.encode([query, answer], convert_to_tensor=True, normalize_embeddings=True)
    return float(st_util.cos_sim(embs[0], embs[1]).cpu().item())

def rougeL_f1(pred: str, ref: str) -> float:
    return _rouge.score(ref, pred)["rougeL"].fmeasure


  Preparing metadata (setup.py) ... done
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=6ead7504f805ed99128947aeedec19309b9b9975201a1107b1562976fd364cd0
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
evaluate_falcon_lora()



Q: What is the importance of unit testing in software engineering?
A: Unit testing is important in software engineering because it helps to ensure that individual units of code are working as intended and that the overall system is functioning correctly. By testing each unit separately, developers can catch errors and bugs early in the development process, which can save time and money in the long run. Additionally, unit testing can help to improve code quality and maintainability by encouraging developers to write code that is more modular and reusable.


Q: List two best practices for clean code.
A: 1. Keep your code organized by using clear variable and function names.
2. Follow the Single Responsibility Principle by separating concerns into distinct modules.


=== Evaluation Report ===
groundedness_avg:      0.000
relevance_avg:         0.664
context_coverage_avg:  0.000
rougeL_f1_avg_on_gold: 0.176
answer_len_avg:        51.0
latency p50 / p95:     4.93s / 7.51s


In [ ]:
import json, textwrap

def _get_gen(max_new_tokens):
    # Use your existing pipeline loader
    return load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)

# ---------------------------
# 1) Ask a custom question
# ---------------------------
def ask_falcon(question: str,
               context: str = None,
               system: str = "You are a precise, concise AI assistant.",
               max_new_tokens: int = 384,
               do_sample: bool = False,
               temperature: float = 0.7,
               top_p: float = 0.9) -> str:
    """
    Ask Falcon(+LoRA) a question.
    If `context` is given, the assistant is instructed to only use that context.
    """
    gen = _get_gen(max_new_tokens)

    if context:
        prompt = f"""{system}

Use ONLY the provided context to answer. If the answer is not in the context, say "Not enough information."

Context:
{context}

Question: {question}
Answer:"""
    else:
        prompt = f"""{system}

Question: {question}
Answer:"""

    if do_sample:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                  temperature=temperature, top_p=top_p, repetition_penalty=1.1)
    else:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False,
                  repetition_penalty=1.1)

    return out[0]["generated_text"][len(prompt):].strip()


# ---------------------------
# 2) Summarize text
# ---------------------------
def summarize_falcon(text: str,
                     n_sentences: int = 3,
                     style: str = "concise",
                     system: str = "You are a precise, concise AI assistant.",
                     max_new_tokens: int = 384,
                     do_sample: bool = False,
                     temperature: float = 0.7,
                     top_p: float = 0.9) -> str:
    """
    Summarize the given text into ~n_sentences with the requested style.
    """
    gen = _get_gen(max_new_tokens)
    text = textwrap.dedent(text).strip()

    prompt = f"""{system}

Write a {style} summary of about {n_sentences} sentences using ONLY the provided text.
If important details are missing, say "Not enough information."

Text:
{text}

Summary:"""

    if do_sample:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                  temperature=temperature, top_p=top_p, repetition_penalty=1.1)
    else:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False,
                  repetition_penalty=1.1)

    return out[0]["generated_text"][len(prompt):].strip()


# ---------------------------------------
# 3) Create MCQs from source text/topic
# ---------------------------------------
def create_mcq_falcon(source: str,
                      n: int = 5,
                      system: str = "You are a careful exam item writer.",
                      max_new_tokens: int = 768,
                      do_sample: bool = False,
                      temperature: float = 0.7,
                      top_p: float = 0.9,
                      json_output: bool = True):
    """
    Create MCQs derived STRICTLY from the provided source string.
    If json_output=True, returns a Python list of dicts parsed from JSON.
    Otherwise returns the raw model string.
    """
    gen = _get_gen(max_new_tokens)
    source = textwrap.dedent(source).strip()

    if json_output:
        format_instructions = (
            'Return ONLY valid JSON: a list of objects with keys '
            '["question","options","answer","rationale"]. '
            '"options" must be a list of 4 strings labeled A–D in the text, '
            '"answer" must be one of ["A","B","C","D"]. No markdown, no prose.'
        )
    else:
        format_instructions = (
            "Output each item as:\n"
            "Q: ...\nA) ...\nB) ...\nC) ...\nD) ...\nAnswer: <A|B|C|D>\nWhy: <1-line rationale>\n\n"
        )

    prompt = f"""{system}

You will write {n} high-quality single-best-answer MCQs STRICTLY from the source below.
Each item must have:
- Question (clear, unambiguous)
- Four options (A–D), plausible distractors, one correct answer
- A 1-line rationale explaining the correct answer
Do NOT invent facts not present in the source.

{("Formatting: " + format_instructions) if json_output else ("Formatting:\n" + format_instructions)}

Source:
{source}

MCQs:"""

    if do_sample:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=True,
                  temperature=temperature, top_p=top_p, repetition_penalty=1.1)
    else:
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False,
                  repetition_penalty=1.1)

    text_out = out[0]["generated_text"][len(prompt):].strip()

    if not json_output:
        return text_out

    # Try to parse JSON. If it fails, return the raw text for inspection.
    try:
        # Extract JSON blob if model added leading/trailing prose
        start = text_out.find('[')
        end = text_out.rfind(']')
        if start != -1 and end != -1:
            text_out = text_out[start:end+1]
        data = json.loads(text_out)
        # minimal sanity check
        assert isinstance(data, list) and all(isinstance(x, dict) for x in data)
        return data
    except Exception as e:
        return {"raw": text_out, "parse_error": str(e)}


In [ ]:
# 1) Ask
print(ask_falcon("What is proofreading?"))

# # 2) Summarize
# long_text = "Low-Rank Adaptation (LoRA) injects small trainable low-rank matrices into frozen weights..."
# print(summarize_falcon(long_text, n_sentences=2))

# # 3) MCQs
# src = """LoRA reduces trainable parameters by decomposing weight updates into low-rank factors.
# It keeps the base model frozen, improving memory and training efficiency."""
# mcqs = create_mcq_falcon(src, n=3, json_output=True)
# print(mcqs)  # list of dicts (or dict with 'raw' + 'parse_error' if JSON parse failed)


Proofreading is the process of carefully checking a document or text for errors and mistakes. It involves reading the text aloud, checking for typos, grammar errors, and other mistakes that may have been missed during the initial writing process. Proofreading is an essential step in the writing process, as it ensures that the final product is error-free and professional.


In [ ]:
# =========================
# RAG: build index & retriever
# =========================
!pip -q install langchain langchain-community sentence-transformers faiss-cpu pypdf

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# --- Build from your PDF(s) ---
PDF_PATH = "/content/drive/MyDrive/AI/train_your_bot.pdf"   # <= set this
loader = PyPDFLoader(PDF_PATH)
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=700, chunk_overlap=120)
docs = splitter.split_documents(documents)
for d in docs:
    d.page_content = d.page_content.replace("\n", " ").strip()
print("[RAG] Total chunks:", len(docs))

embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(docs, embed_model)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

def retrieve_context(query: str, k: int = 3):
    retriever.search_kwargs["k"] = k
    src_docs = retriever.invoke(query if query else "summary")
    ctx = "\n\n".join([d.page_content for d in src_docs])
    return ctx, src_docs

# =========================
# Grounded ask/summarize/MCQ using retrieved context
# =========================
def ask_falcon_rag(question: str, k: int = 3, max_new_tokens: int = 384):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)
    context, srcs = retrieve_context(question, k=k)
    prompt = (
        "You are an assistant that answers ONLY using the provided context.\n"
        "If the answer is not in the context, say 'Not enough information.'\n\n"
        f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
    )
    out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    ans = out[0]["generated_text"][len(prompt):].strip()
    return ans, srcs

def summarize_falcon_rag(topic: str, k: int = 5, n_sentences: int = 3, max_new_tokens: int = 384):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)
    context, srcs = retrieve_context(topic, k=k)
    prompt = (
        "You are an assistant that writes concise summaries ONLY using the provided context.\n"
        f"Write a summary of about {n_sentences} sentences focused strictly on: \"{topic}\".\n"
        "If insufficient info, say 'Not enough information.'\n\n"
        f"Context:\n{context}\n\nSummary:"
    )
    out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    ans = out[0]["generated_text"][len(prompt):].strip()
    return ans, srcs

def create_mcq_falcon_rag(topic: str, n: int = 5, k: int = 6, max_new_tokens: int = 768):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)
    context, srcs = retrieve_context(topic, k=k)
    prompt = (
        "You are an assistant that generates MCQs ONLY from the provided context.\n"
        f"Create {n} high-quality MCQs that focus strictly on: \"{topic}\".\n"
        "Each item MUST include:\n- Question\n- Options (A–D)\n- Correct Answer (single best)\n- 1-line Rationale\n"
        "If there is not enough information to make MCQs, say 'Not enough information.'\n\n"
        f"Context:\n{context}\n\nMCQs:"
    )
    out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
    mcqs = out[0]["generated_text"][len(prompt):].strip()
    return mcqs, srcs

# =========================
# Metrics (same as before) + grounded eval
# =========================
from sentence_transformers import SentenceTransformer, util as st_util
from rouge_score import rouge_scorer
import re, time, numpy as np
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
_rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def _normalize(s): s=s.lower(); s=re.sub(r"[^a-z0-9\s]"," ",s); return re.sub(r"\s+"," ",s).strip()
def _tokens(s): return set(_normalize(s).split())
def groundedness(answer, source_docs):
    ctx = " ".join([d.page_content for d in source_docs])
    ans_t, ctx_t = _tokens(answer), _tokens(ctx)
    return 0.0 if not ans_t else len(ans_t & ctx_t) / len(ans_t)
def context_coverage(answer, source_docs, cap=8000):
    ctx = " ".join([d.page_content[:cap] for d in source_docs])
    ans_t, ctx_t = _tokens(answer), _tokens(ctx)
    return 0.0 if not ctx_t else len(ctx_t & ans_t) / len(ctx_t)
def relevance(query, answer):
    embs = embedder.encode([query, answer], convert_to_tensor=True, normalize_embeddings=True)
    return float(st_util.cos_sim(embs[0], embs[1]).cpu().item())
def rougeL_f1(pred, ref): return _rouge.score(ref, pred)["rougeL"].fmeasure

# Replace with your real eval set / golds for your PDF
EVAL_QUERIES = [
    "Give a 2–3 sentence summary of the PDF.",
    "List three key practices recommended in the PDF and explain each briefly.",
    "What is the main concept of proofreading according to this PDF?",
    "Name two pitfalls the document warns about and how to avoid them.",
    "Provide a short checklist derived strictly from the PDF."
]
GOLD = {
    "What is the main concept of proofreading according to this PDF?":
        "Proofreading is the final review focused on surface-level errors such as spelling, punctuation and formatting/layout.",
    "Provide a short checklist derived strictly from the PDF.":
        "Check headers and page numbers; verify references match citations; apply a style guide for capitalization and hyphenation; scan for spelling and punctuation errors; ensure consistent layout."
}

def evaluate_falcon_lora_rag(k:int=3, max_new_tokens:int=384):
    gen = load_falcon_lora_pipeline(max_new_tokens=max_new_tokens)
    latencies=[]; g_list=[]; r_list=[]; c_list=[]; lens=[]; rouge_list=[]
    for q in EVAL_QUERIES:
        ctx, srcs = retrieve_context(q, k=k)
        prompt = (
            "You are an assistant that answers ONLY using the provided context.\n"
            "If the answer is not in the context, say 'Not enough information.'\n\n"
            f"Context:\n{ctx}\n\nQuestion: {q}\n\nAnswer:"
        )
        t0 = time.time()
        out = gen(prompt, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1)
        t1 = time.time()
        ans = out[0]["generated_text"][len(prompt):].strip()

        g = groundedness(ans, srcs)
        r = relevance(q, ans)
        c = context_coverage(ans, srcs)
        L = len(ans.split())
        rl = rougeL_f1(ans, GOLD[q]) if q in GOLD else None

        latencies.append(t1-t0); g_list.append(g); r_list.append(r); c_list.append(c); lens.append(L)
        if rl is not None: rouge_list.append(rl)

    def F(x,s):
        try: return format(float(np.mean(x)), s)
        except: return "None"

    lat_p50 = float(np.percentile(latencies, 50)) if latencies else None
    lat_p95 = float(np.percentile(latencies, 95)) if latencies else None
    print("=== Evaluation Report (RAG) ===")
    print("groundedness_avg:     ", F(g_list, ".3f"))
    print("relevance_avg:        ", F(r_list, ".3f"))
    print("context_coverage_avg: ", F(c_list, ".3f"))
    print("rougeL_f1_avg_on_gold:", (F(rouge_list, ".3f") if rouge_list else "-"))
    print("answer_len_avg:       ", F(lens, ".1f"))
    p50s = (format(lat_p50, ".2f")+"s") if lat_p50 is not None else "None"
    p95s = (format(lat_p95, ".2f")+"s") if lat_p95 is not None else "None"
    print("latency p50 / p95:    ", p50s, "/", p95s)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.5/310.5 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


[RAG] Total chunks: 628


/tmp/ipython-input-164287197.py:22: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embed_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


In [ ]:
# Grounded Q&A
ans, srcs = ask_falcon_rag("What is proofreading according to the PDF?", k=3)
print(ans)

# Grounded summary
summary, _ = summarize_falcon_rag("proofreading", k=5, n_sentences=3)
print(summary)

# Grounded MCQs
mcqs, _ = create_mcq_falcon_rag("proofreading", n=5, k=6)
print(mcqs)

# Grounded evaluation (non-zero groundedness/context_coverage)
evaluate_falcon_lora_rag(k=3, max_new_tokens=384)


Proofreading is the final stage of the editing process, focusing on surface errors such as misspellings and mistakes in grammar and punctuation. You should proofread only after you have finished all of your other editing revisions. Why proofread? It's the content that really matters, right? Content is important. But like it or not, the way a paper looks affects the way others judge it. When you've worked hard to develop and present your ideas, you don't want careless errors distracting your reader from what you have to say. It's worth paying attention to the details that help you to make a good impression.
Proofreading is the final stage of the editing process, focusing on surface errors such as misspellings and mistakes in grammar and punctuation. You should proofread only after you have finished all of your other editing revisions. Why proofread? It's the content that really matters, right? Content is important. But like it or not, the way a paper looks affects the way others judge i

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


=== Evaluation Report (RAG) ===
groundedness_avg:      0.511
relevance_avg:         0.487
context_coverage_avg:  0.189
rougeL_f1_avg_on_gold: 0.139
answer_len_avg:        117.2
latency p50 / p95:     8.30s / 23.87s


In [ ]:
summary, _ = summarize_falcon_rag("proofreading", k=5, n_sentences=3)
print(summary)

Proofreading is the final stage of the editing process, focusing on surface errors such as misspellings and mistakes in grammar and punctuation. You should proofread only after you have finished all of your other editing revisions. Why proofread? It's the content that really matters, right? Content is important. But like it or not, the way a paper looks affects the way others judge it. When you've worked hard to develop and present your ideas, you don't want careless errors distracting your reader from what you have to say. It's worth paying attention to the details that help you to make a good impression.
